### Data Cleaning

#### Cleaning the data to make a dataset for EDA, feature engineering and modelling

#### Importing Packages and Loading Data

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

In [24]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 50)

In [25]:
project_root = Path(".").resolve().parent
raw_path = project_root/"data"/"raw"/"pp-2025.csv"
processed_path = project_root / "data" / "processed" / "london_clean.parquet"

In [26]:
# Column Names

PP_COLUMNS = [
    "transaction_id", # Unique ID for transactions
    "price", # Sales Price in £ (Target Variable)
    "date_of_transfer", # Date of Completion
    "postcode", # Full Postcode
    "property_type", # D=Detached, S=Semi, T=Terraced, O=Other
    "old_new", # Y=New Building, N=Existing
    "duration", # F=Freehold, L=Leasehold, U=Unknown
    "paon", # Primary Address (Number, Building Name)
    "saon", # Secondary Address (Flat, Unit)
    "street", # Street name
    "locality", # District Name, Locality
    "town_city", # Town or City
    "district", # Local Authority District (example: London Borough)
    "county", # County Name
    "ppd_category", # A=Standard Sale, B=Non-Standard
    "record_status" # A=Addition, C=Change, D=Deletion
]

In [27]:
df_raw = pd.read_csv(
    raw_path,
    header=None,
    names=PP_COLUMNS,
    encoding="latin-1"
)

In [29]:
df_raw.head()

,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_status
0,{49E87C31-9D5F-591C-E063-4704A8C00C31},560000,04/12/2025 00:00,PE1 2QU,D,N,F,10,NaN,ALL SAINTS ROAD,NaN,PETERBOROUGH,CITY OF PETERBOROUGH,CITY OF PETERBOROUGH,A,A
1,{49E87C31-9D60-591C-E063-4704A8C00C31},230000,12/12/2025 00:00,PE2 9RY,T,N,F,19,NaN,OSWALD ROAD,NaN,PETERBOROUGH,CITY OF PETERBOROUGH,CITY OF PETERBOROUGH,A,A
2,{49E87C31-9D61-591C-E063-4704A8C00C31},272000,15/12/2025 00:00,PE16 6BL,D,N,F,2,NaN,SADDLERS WAY,NaN,CHATTERIS,FENLAND,CAMBRIDGESHIRE,A,A
3,{49E87C31-9D63-591C-E063-4704A8C00C31},249950,17/12/2025 00:00,CB4 3RY,T,N,L,72,NaN,COCKERELL ROAD,NaN,CAMBRIDGE,CAMBRIDGE,CAMBRIDGESHIRE,A,A
4,{49E87C31-9D65-591C-E063-4704A8C00C31},650000,12/12/2025 00:00,CB1 3BH,T,N,F,21,NaN,CAVENDISH PLACE,NaN,CAMBRIDGE,CAMBRIDGE,CAMBRIDGESHIRE,A,A


#### Filtering to London

In [30]:
df_lon = df_raw[df_raw["county"] == "GREATER LONDON"]
# London is our project
df_lon.head()

,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_status
192,{44F406B7-3032-1095-E063-4704A8C048D4},200000,16/10/2025 00:00,IG6 2DZ,F,N,L,114A,NaN,HIGH STREET,BARKINGSIDE,ILFORD,REDBRIDGE,GREATER LONDON,B,A
193,{44F406B7-3033-1095-E063-4704A8C048D4},420000,20/01/2025 00:00,E2 7EL,F,N,L,KARSLAKE HOUSE,FLAT 5,GIBRALTAR WALK,NaN,LONDON,TOWER HAMLETS,GREATER LONDON,B,A
194,{44F406B7-3035-1095-E063-4704A8C048D4},662000,23/10/2025 00:00,IG1 4LB,T,N,F,28,NaN,AIRLIE GARDENS,NaN,ILFORD,REDBRIDGE,GREATER LONDON,B,A
195,{44F406B7-3036-1095-E063-4704A8C048D4},184000,31/01/2025 00:00,RM1 1AR,F,N,L,9,NaN,MALT HOUSE PLACE,NaN,ROMFORD,HAVERING,GREATER LONDON,B,A
196,{44F406B7-3038-1095-E063-4704A8C048D4},575000,07/11/2025 00:00,E14 0JU,F,N,L,42 - 44,UNIT 14,ORCHARD PLACE,NaN,LONDON,TOWER HAMLETS,GREATER LONDON,B,A


In [36]:
# Filtering to PPD Category A: Standard market sales
df_a = df_lon[df_lon["ppd_category"] == "A"]

# Category B sales include repossessions, portfolio sales,
# transfers between related parties, and other non-standard transactions whose
# prices don't reflect open-market value. Our model targets fair market prices.

df_a

,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_status
326,{49E87C31-854D-591C-E063-4704A8C00C31},215000,31/01/2025 00:00,E3 2PQ,F,N,L,"MOCHA COURT, 14",FLAT 17,TAYLOR PLACE,NaN,LONDON,TOWER HAMLETS,GREATER LONDON,A,A
327,{49E87C31-854E-591C-E063-4704A8C00C31},349000,22/12/2025 00:00,E17 7LB,F,N,L,"HENLEY LODGE, 2",FLAT 28,WILLOW WALK,NaN,LONDON,WALTHAM FOREST,GREATER LONDON,A,A
328,{49E87C31-854F-591C-E063-4704A8C00C31},540000,16/12/2025 00:00,NW4 3PG,S,N,F,1A,NaN,DANIEL PLACE,NaN,LONDON,BARNET,GREATER LONDON,A,A
329,{49E87C31-8550-591C-E063-4704A8C00C31},460000,15/12/2025 00:00,E2 8FZ,F,N,L,SOUTH MILL APARTMENTS,FLAT 12,HEBDEN STREET,NaN,LONDON,HACKNEY,GREATER LONDON,A,A
330,{49E87C31-8552-591C-E063-4704A8C00C31},265000,19/12/2025 00:00,E14 9BF,F,N,L,"CHARRINGTON TOWER, 11",APARTMENT 2502,BISCAYNE AVENUE,NaN,LONDON,TOWER HAMLETS,GREATER LONDON,A,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
801188,{38EDC0C2-DDAF-0F63-E063-4704A8C00424},393800,11/06/2025 00:00,E10 7PE,F,N,L,103,NaN,KETTLEBASTON ROAD,NaN,LONDON,WALTHAM FOREST,GREATER LONDON,A,A
801441,{38EDC0C2-DDF8-0F63-E063-4704A8C00424},388000,29/05/2025 00:00,RM13 9SN,S,N,F,53,NaN,LAKESIDE,NaN,RAINHAM,HAVERING,GREATER LONDON,A,A
801953,{38EDC0C2-DE93-0F63-E063-4704A8C00424},358000,05/06/2025 00:00,RM3 7DX,T,N,F,9,NaN,GRANGE ROAD,NaN,ROMFORD,HAVERING,GREATER LONDON,A,A
801961,{38EDC0C2-DE9E-0F63-E063-4704A8C00424},1095000,03/01/2025 00:00,E18 2PS,S,N,F,50,NaN,DERBY ROAD,NaN,LONDON,REDBRIDGE,GREATER LONDON,A,A


#### Dropping rows with missing postcodes

In [37]:
df_a = df_a[df_a["postcode"].notna()]

df_a

,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_status
326,{49E87C31-854D-591C-E063-4704A8C00C31},215000,31/01/2025 00:00,E3 2PQ,F,N,L,"MOCHA COURT, 14",FLAT 17,TAYLOR PLACE,NaN,LONDON,TOWER HAMLETS,GREATER LONDON,A,A
327,{49E87C31-854E-591C-E063-4704A8C00C31},349000,22/12/2025 00:00,E17 7LB,F,N,L,"HENLEY LODGE, 2",FLAT 28,WILLOW WALK,NaN,LONDON,WALTHAM FOREST,GREATER LONDON,A,A
328,{49E87C31-854F-591C-E063-4704A8C00C31},540000,16/12/2025 00:00,NW4 3PG,S,N,F,1A,NaN,DANIEL PLACE,NaN,LONDON,BARNET,GREATER LONDON,A,A
329,{49E87C31-8550-591C-E063-4704A8C00C31},460000,15/12/2025 00:00,E2 8FZ,F,N,L,SOUTH MILL APARTMENTS,FLAT 12,HEBDEN STREET,NaN,LONDON,HACKNEY,GREATER LONDON,A,A
330,{49E87C31-8552-591C-E063-4704A8C00C31},265000,19/12/2025 00:00,E14 9BF,F,N,L,"CHARRINGTON TOWER, 11",APARTMENT 2502,BISCAYNE AVENUE,NaN,LONDON,TOWER HAMLETS,GREATER LONDON,A,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
801188,{38EDC0C2-DDAF-0F63-E063-4704A8C00424},393800,11/06/2025 00:00,E10 7PE,F,N,L,103,NaN,KETTLEBASTON ROAD,NaN,LONDON,WALTHAM FOREST,GREATER LONDON,A,A
801441,{38EDC0C2-DDF8-0F63-E063-4704A8C00424},388000,29/05/2025 00:00,RM13 9SN,S,N,F,53,NaN,LAKESIDE,NaN,RAINHAM,HAVERING,GREATER LONDON,A,A
801953,{38EDC0C2-DE93-0F63-E063-4704A8C00424},358000,05/06/2025 00:00,RM3 7DX,T,N,F,9,NaN,GRANGE ROAD,NaN,ROMFORD,HAVERING,GREATER LONDON,A,A
801961,{38EDC0C2-DE9E-0F63-E063-4704A8C00424},1095000,03/01/2025 00:00,E18 2PS,S,N,F,50,NaN,DERBY ROAD,NaN,LONDON,REDBRIDGE,GREATER LONDON,A,A


#### Filtering Price Range
Our profiling found a minimum price of £1 and maximum price of £793,020,000. These are not normal residential market transactions.
The £1 entries are token value family transfers and the £100M+ entries are corporate sales. Keeping price range between £50k and £10M is realistic for London residential sales while excluding the extreme outliers that would affect our model.



In [40]:
price_min = 50000
price_max = 10000000

df_a = df_a[(df_a["price"] >= price_min) & (df_a["price"] <= price_max)]

df_a

,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_status
326,{49E87C31-854D-591C-E063-4704A8C00C31},215000,31/01/2025 00:00,E3 2PQ,F,N,L,"MOCHA COURT, 14",FLAT 17,TAYLOR PLACE,NaN,LONDON,TOWER HAMLETS,GREATER LONDON,A,A
327,{49E87C31-854E-591C-E063-4704A8C00C31},349000,22/12/2025 00:00,E17 7LB,F,N,L,"HENLEY LODGE, 2",FLAT 28,WILLOW WALK,NaN,LONDON,WALTHAM FOREST,GREATER LONDON,A,A
328,{49E87C31-854F-591C-E063-4704A8C00C31},540000,16/12/2025 00:00,NW4 3PG,S,N,F,1A,NaN,DANIEL PLACE,NaN,LONDON,BARNET,GREATER LONDON,A,A
329,{49E87C31-8550-591C-E063-4704A8C00C31},460000,15/12/2025 00:00,E2 8FZ,F,N,L,SOUTH MILL APARTMENTS,FLAT 12,HEBDEN STREET,NaN,LONDON,HACKNEY,GREATER LONDON,A,A
330,{49E87C31-8552-591C-E063-4704A8C00C31},265000,19/12/2025 00:00,E14 9BF,F,N,L,"CHARRINGTON TOWER, 11",APARTMENT 2502,BISCAYNE AVENUE,NaN,LONDON,TOWER HAMLETS,GREATER LONDON,A,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
801188,{38EDC0C2-DDAF-0F63-E063-4704A8C00424},393800,11/06/2025 00:00,E10 7PE,F,N,L,103,NaN,KETTLEBASTON ROAD,NaN,LONDON,WALTHAM FOREST,GREATER LONDON,A,A
801441,{38EDC0C2-DDF8-0F63-E063-4704A8C00424},388000,29/05/2025 00:00,RM13 9SN,S,N,F,53,NaN,LAKESIDE,NaN,RAINHAM,HAVERING,GREATER LONDON,A,A
801953,{38EDC0C2-DE93-0F63-E063-4704A8C00424},358000,05/06/2025 00:00,RM3 7DX,T,N,F,9,NaN,GRANGE ROAD,NaN,ROMFORD,HAVERING,GREATER LONDON,A,A
801961,{38EDC0C2-DE9E-0F63-E063-4704A8C00424},1095000,03/01/2025 00:00,E18 2PS,S,N,F,50,NaN,DERBY ROAD,NaN,LONDON,REDBRIDGE,GREATER LONDON,A,A


#### Converting Date

In [41]:
df_a["date_of_transfer"] = pd.to_datetime(df_a["date_of_transfer"], format="%d/%m/%Y %H:%M")

In [43]:
df_a.dtypes

transaction_id                 str
price                        int64
date_of_transfer    datetime64[us]
postcode                       str
property_type                  str
old_new                        str
duration                       str
paon                           str
saon                           str
street                         str
locality                       str
town_city                      str
district                       str
county                         str
ppd_category                   str
record_status                  str
dtype: object